---
## Task 5 - Content Rating & Genre Analysis
---

In [1]:
#Loading the dataset
import pandas as pd
df = pd.read_csv('cleaned_dataset.csv')

### Analyzing Rating Categories


In [2]:
#Inspecting unique ratings and their total counts
rating_counts = df['rating'].value_counts()
print("Rating Counts:")
print(rating_counts)

Rating Counts:
rating
TV-MA       3205
TV-14       2157
TV-PG        861
R            799
PG-13        490
TV-Y7        333
TV-Y         306
PG           287
TV-G         220
NR            79
G             41
TV-Y7-FV       6
NC-17          3
UR             3
Name: count, dtype: int64


### Comparing Ratings Across Content Types

In [3]:
#Defining rating category mapping
rating_map = {
    'TV-MA': 'Adult / Mature', 'R': 'Adult / Mature', 'NC-17': 'Adult / Mature', 'UR': 'Adult / Mature',
    'TV-14': 'Teens / Young Adults', 'PG-13': 'Teens / Young Adults',
    'TV-PG': 'Kids & Family', 'PG': 'Kids & Family', 'TV_Y7': 'Kids & Family',
    'TV-Y7-FV': 'Kids & Family', 'TV-Y': 'Kids & Family', 'TV-G': 'Kids & Family', 'G': 'Kids & Family',
    'NR': 'Unrated'
}

#Applying mapping to DataFrame
df['target_audience'] = df['rating'].map(rating_map)

#Grouping by Target Audience and Content Type
audience_type = df.groupby(['target_audience', 'type'], observed=False).size().unstack(fill_value=0)

display(audience_type)

type,Movie,Tv Show
target_audience,,
Adult / Mature,2865,1145
Kids & Family,1130,591
Teens / Young Adults,1917,730
Unrated,75,4


### Identifying Most Common Genres

In [4]:
df_genres = df.assign(genre=df['listed_in'].str.split(',')).explode('genre')
df_genres['genre'] = df_genres['genre'].str.strip()

#Counting Total occurences per unique genre
genre_counts = df_genres['genre'].value_counts()

#Creating top15 Genres DataFrame
df_top_genres = pd.DataFrame({
    'Genre': genre_counts.head(15).index,
    'Total_Titles': genre_counts.head(15).values
})

#Displaying Top 15 genres and totsl unique genre count
print(f"Total Unique Genres across Platform: {df_genres['genre'].nunique()}")
display(df_top_genres)

Total Unique Genres across Platform: 42


,Genre,Total_Titles
0,International Movies,2752
1,Dramas,2426
2,Comedies,1674
3,International TV Shows,1349
4,Documentaries,869
5,Action & Adventure,859
6,TV Dramas,762
7,Independent Movies,756
8,Children & Family Movies,641
9,Romantic Movies,616


### Interactive Visualizations

In [5]:
#Importing visualization libraries
import plotly.express as px

#Reshaping data for the interactive chart
audience_reset = audience_type.reset_index().melt(
    id_vars='target_audience',
    var_name='Type',
    value_name='Count'
)

#Defining colors
colors = ['#E50914', '#221F1F']

#Creating an interactive grouped bar chart
fig = px.bar(
    audience_reset,
    x='target_audience',
    y='Count',
    color='Type',
    barmode='group',
    text='Count',
    color_discrete_sequence=colors,
    title='Content Distribution by Target Audience and Type',
    labels={
        'target_audience': 'Target Audience Category',
        'Count': 'Number of Titles',
        'Type': 'Content Type'
    }
)

#Adding readable labels and hover details
fig.update_traces(
    texttemplate='%{text:,.0f}',
    textposition='outside',
    cliponaxis=False,
    hovertemplate=(
        '<b>%{x}</b><br>'
        'Content Type: %{fullData.name}<br>'
        'Number of Titles: %{y:,.0f}'
        '<extra></extra>'
    )
)
fig.update_layout(
    template='plotly_white',
    title_x=0.5,
    height=600,
    legend_title_text='Content Type'
)

#Displaying the interactive chart
fig.show()

In [6]:
import plotly.express as px

# Sorting genres from smallest to largest.
# Plotly's horizontal orientation will then display
# the largest genre at the top.
df_plot = df_top_genres.sort_values('Total_Titles', ascending=True)

#Creating interactive horizontal bar chart for top 15 genres
fig = px.bar(
    df_plot,
    x='Total_Titles',
    y='Genre',
    orientation='h',
    text='Total_Titles',
    title='Top 15 Most Common Genres on Netflix'
)

# Setting bar colors
fig.update_traces(
    marker_color=[
        '#E50914' if genre == df_top_genres.loc[
            df_top_genres['Total_Titles'].idxmax(), 'Genre'
        ] else '#221F1F'
        for genre in df_plot['Genre']
    ],
    texttemplate='%{text:,}',
    textposition='outside',
    hovertemplate=(
        '<b>%{y}</b><br>'
        'Number of titles: %{x:,}'
        '<extra></extra>'
    )
)

# Adding layout details for better readability
fig.update_layout(
    xaxis_title='Number of titles',
    yaxis_title='Genre',
    xaxis_range=[
        0,
        df_plot['Total_Titles'].max() * 1.12
    ],
    height=600,
    template='plotly_white',
    title={
        'text': 'Top 15 Most Common Genres on Netflix',
        'font': {'size': 20}
    }
)

# Displaying the interactive chart
fig.show()

### Business Insights
1. **Dominance of Mature Content:**
* Mature Content ('Adult / Mature' and 'Teens / Young Adults') accounts for over **60%** of Netflix's entire library, led by **TV-MA** (3,205 titles) and **TV-14** (2,157 titles).
* Movies lean heavily into mature and young adult ratings, while TV Shows have a higher proportion for family an dkids programming ('TV-Y', 'TV-Y7', -TV-PG').

2. **Core Content Pillars (Top Genres):**
* **International Content & Regional Dramas:** 'International Movies' (2,752) and 'Dramas" (2,426) are the top two overall genres, reflectingNetflix's strategy of acquired localized international content with broad global appeal.
* **Comedies & Documentaries:** 'Comedies' (1,674) and 'Documemtaries' (869) complete the core engagement foundation for subscriber acquisition and retention.

3. **Strategic Recommendations:**
* Continue Investing in international local-language dramas to drive subscribers in emerging regional markets, while maintaining a steady catalog of family/kids content to reduce householder subscriber churn.